# Qualidade de Software em Python — Tutorial

## Objetivos

Ao final deste tutorial você será capaz de:

- Reconhecer e aplicar PEP 8 / Zen of Python em código próprio.
- Escrever docstrings conforme PEP 257.
- Anotar funções com type hints (PEP 484/585/604) e validar com `mypy`.
- Definir e usar `Protocol` para subtipagem estrutural (PEP 544).
- Usar encadeamento, grupos e notas de exceção (PEP 3134/654/678).
- Estruturar um projeto com `pyproject.toml` (PEP 517/518/621).

## 1. Legibilidade: PEP 8 e o Zen of Python

PEP 8 define convenções visuais (indentação de 4 espaços, `snake_case` em funções, `CamelCase` em classes, espaços ao redor de operadores). O Zen (`import this`) orienta decisões de design.

In [ ]:
# Exemplo: ler o Zen of Python (PEP 20)
import this

In [ ]:
# Exercicio 1: reescreva a funcao abaixo seguindo PEP 8
# - nomes snake_case
# - espacos ao redor de operadores e apos virgulas
# - 4 espacos de indentacao

def CalcMedia(L ):
  s=0
  for X in L:s=s+X
  return s/len( L )

# TODO: reescreva como calc_media(lista) abaixo
def calc_media(lista):
    pass

assert calc_media([1, 2, 3, 4]) == 2.5, 'Verifique sua implementacao'

## 2. Documentação: PEP 257

Docstrings com triplo aspas, sumário em uma linha, descrição opcional e seções para parâmetros, retorno e exceções.

In [ ]:
# Exemplo: docstring estilo reST (PEP 287)
def fatorial(n: int) -> int:
    """Calcula n! de forma iterativa.

    :param n: inteiro nao-negativo.
    :returns: o fatorial de n.
    :raises ValueError: se n for negativo.
    """
    if n < 0:
        raise ValueError('n deve ser >= 0')
    r = 1
    for k in range(2, n + 1):
        r *= k
    return r

print(fatorial.__doc__)
print('5! =', fatorial(5))

In [ ]:
# Exercicio 2: adicione uma docstring PEP 257 + PEP 287 a esta funcao

def media_ponderada(valores, pesos):
    # TODO: escreva docstring com sumario, :param:, :returns:, :raises:
    if len(valores) != len(pesos):
        raise ValueError('tamanhos diferentes')
    return sum(v * p for v, p in zip(valores, pesos)) / sum(pesos)

assert media_ponderada.__doc__ is not None, 'Adicione a docstring'
assert abs(media_ponderada([1, 2, 3], [1, 1, 2]) - 2.25) < 1e-9

## 3. Tipagem estática: PEP 484, 585, 604, 544

Anotações de tipo são opcionais em tempo de execução, mas verificáveis com `mypy` ou `pyright`. Desde Python 3.9/3.10 podemos usar `list[int]` e `X | None` sem importar de `typing`.

In [ ]:
# Exemplo: sintaxe moderna (PEP 585 + 604)
def primeiro_pares(numeros: list[int]) -> int | None:
    """Devolve o primeiro numero par, ou None se nao houver."""
    for n in numeros:
        if n % 2 == 0:
            return n
    return None

print(primeiro_pares([1, 3, 4, 5]))
print(primeiro_pares([1, 3, 5]))

In [ ]:
# Exemplo: Protocols (PEP 544) - subtipagem estrutural
from typing import Protocol

class TemArea(Protocol):
    def area(self) -> float: ...

class Circulo:
    def __init__(self, r: float) -> None:
        self.r = r
    def area(self) -> float:
        return 3.14159 * self.r ** 2

class Retangulo:
    def __init__(self, b: float, h: float) -> None:
        self.b, self.h = b, h
    def area(self) -> float:
        return self.b * self.h

def soma_areas(figuras: list[TemArea]) -> float:
    return sum(f.area() for f in figuras)

print(soma_areas([Circulo(1), Retangulo(2, 3)]))

In [ ]:
# Exercicio 3: anote esta funcao com PEP 585 + 604 (sem importar de typing)

def agrupar_por_paridade(numeros):
    # TODO: assine como agrupar_por_paridade(numeros: list[int]) -> dict[str, list[int]]
    pares, impares = [], []
    for n in numeros:
        (pares if n % 2 == 0 else impares).append(n)
    return {'pares': pares, 'impares': impares}

res = agrupar_por_paridade([1, 2, 3, 4, 5])
assert res == {'pares': [2, 4], 'impares': [1, 3, 5]}
assert agrupar_por_paridade.__annotations__, 'Adicione as anotacoes de tipo'

## 4. Robustez: PEP 3134, 654, 678

Encadeamento (`raise ... from`), grupos (`ExceptionGroup`/`except*`) e notas (`add_note`) enriquecem o diagnóstico.

In [ ]:
# Exemplo: encadeamento e notas
class ConfigInvalida(Exception):
    pass

def carregar(texto: str) -> dict:
    try:
        chave, valor = texto.split('=')
        return {chave.strip(): int(valor)}
    except ValueError as e:
        e.add_note(f'entrada recebida: {texto!r}')
        raise ConfigInvalida('formato chave=valor invalido') from e

try:
    carregar('porta=abc')
except ConfigInvalida as e:
    print('Exception capturada:', e)
    print('Causa raiz:', repr(e.__cause__))
    print('Notas da causa:', getattr(e.__cause__, '__notes__', []))

In [ ]:
# Exemplo: ExceptionGroup e except* (PEP 654)
erros = ExceptionGroup('falhas no batch', [
    ValueError('linha 1'),
    TimeoutError('linha 2'),
    ValueError('linha 7'),
])

try:
    raise erros
except* ValueError as eg:
    print('ValueErrors:', [str(e) for e in eg.exceptions])
except* TimeoutError as eg:
    print('Timeouts:', [str(e) for e in eg.exceptions])

In [ ]:
# Exercicio 4: anexe uma nota com o indice da linha quando falhar

def parse_inteiros(linhas: list[str]) -> list[int]:
    # TODO: para cada linha, tente int(linha). Se ValueError,
    # use add_note(f'linha {i}: {linha!r}') e relevante o erro com `raise`.
    resultado: list[int] = []
    for i, linha in enumerate(linhas):
        resultado.append(int(linha))
    return resultado

try:
    parse_inteiros(['1', '2', 'tres'])
except ValueError as e:
    assert getattr(e, '__notes__', []), 'Anexe uma nota com add_note'

## 5. Empacotamento: pyproject.toml (PEP 517/518/621)

Um projeto Python moderno declara metadados, dependências e backend de build num único `pyproject.toml`.

In [ ]:
# Exemplo: gerar um pyproject.toml minimo em memoria
exemplo = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "qualidade-demo"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["requests>=2.31,<3"]

[project.optional-dependencies]
dev = ["pytest", "mypy", "ruff"]
'''
print(exemplo)

## Desafio Final

Implemente um módulo `notas.py` que receba uma lista de tuplas `(aluno, nota)` e calcule a média da turma, com qualidade:

1. Função `media_turma(registros: list[tuple[str, float]]) -> float` com anotações de tipo PEP 585/604.
2. Docstring PEP 257 + PEP 287 (com `:param:`, `:returns:`, `:raises:`).
3. Validação: notas devem estar em `[0, 10]`. Se inválida, lance `ValueError` com `add_note` indicando o aluno problemático (PEP 678).
4. Se a lista estiver vazia, lance `ValueError('lista vazia')`.
5. Código aprovado por `ruff check` e `mypy --strict`.

In [ ]:
# Desafio: implemente media_turma abaixo

def media_turma(registros):
    # TODO: sua solucao aqui
    pass

# Testes basicos
assert abs(media_turma([('a', 8.0), ('b', 6.0)]) - 7.0) < 1e-9
try:
    media_turma([])
except ValueError:
    pass
else:
    raise AssertionError('Deveria lancar ValueError para lista vazia')

try:
    media_turma([('x', 11.0)])
except ValueError as e:
    assert getattr(e, '__notes__', []), 'Use add_note com nome do aluno'
else:
    raise AssertionError('Deveria lancar ValueError para nota fora do intervalo')

## Referências

Veja o arquivo `../referencias.bib` para a lista completa.